In [1]:
from numcosmo_py.catalog import MockGenerator
from numcosmo_py import Nc, Ncm
from numcosmo_py.catalog import sky_match
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import unique, Table
from astropy.table import vstack
from  run_confusion_matrix import calculate_split_metrics, calculate_catalog_metrics
from matplotlib import pyplot as plt
Ncm.cfg_init()
%load_ext autoreload
%autoreload 2

In [2]:
Omega_b = 0.0486
Omega_c = 0.2614
Omega_k = 0.0
H0 = 67.7

cosmo = Nc.HICosmoDEXcdm.new()
cosmo.omega_x2omega_k()
cosmo["Omegab"] = Omega_b
cosmo["Omegac"] = Omega_c
cosmo["Omegak"] = Omega_k
cosmo["H0"] = H0
cosmo["w"] = -1.0

prim = Nc.HIPrimPowerLaw.new()
prim.param_set_by_name("ln10e10ASA", 3.0)
prim.props.n_SA =  0.963 

reion = Nc.HIReionCamb.new()

cosmo.add_submodel(prim)
cosmo.add_submodel(reion)

tf = Nc.TransferFuncEH()

psml = Nc.PowspecMLTransfer.new(tf)
psml.require_kmin(1.0e-6)
psml.require_kmax(1.0e3)


psf = Ncm.PowspecFilter.new(psml, Ncm.PowspecFilterType.TOPHAT)
psf.set_best_lnr0()
psf.prepare(cosmo)

old_amplitude = np.exp(prim.props.ln10e10ASA)
ln10e10ASA = np.log((0.8/ cosmo.sigma8(psf)) ** 2 * old_amplitude)
prim.param_set_desc("ln10e10ASA", {"lower-bound": 2.0,"upper-bound": 5.0,"scale": 1.0,"abstol": 1.0e-50,"fit": True,"value": float(ln10e10ASA)})

dist = Nc.Distance.new(2.0)
dist.prepare(cosmo)

mulf = Nc.MultiplicityFuncDespali.new()
mulf.set_mdef(Nc.MultiplicityFuncMassDef.VIRIAL)
hmf = Nc.HaloMassFunction.new(dist, psf, mulf)

cluster_m = Nc.ClusterMassAscaso(lnRichness_min = np.log(1) ,lnRichness_max = np.log(1000), M0=1e13)
cluster_m.param_set_by_name("cut", np.log(1))
cluster_m.param_set_desc("mup0",{"fit": True,"value": 3.022142935})
cluster_m.param_set_desc("mup1",{"fit": True,"value": 1.25})
cluster_m.param_set_desc("mup2",{"fit": True,"value": 0.0})
cluster_m.param_set_desc("sigmap0",{"fit": True,"value": 0.5})
cluster_m.param_set_desc("sigmap1",{"fit": True,"value": 0.0})
cluster_m.param_set_desc("sigmap2",{"fit": True,"value": 0.0})
cluster_m = Nc.ClusterMassLnnormal(lnMobs_min=np.log(1e13), lnMobs_max=np.log(1e16),bias=0,sigma=0.1)

In [3]:
class Completeness_model:
    
    """Class to generate completeness model for halo/cluster mocks."""
    def __init__(self, a):
        self.a = a

    
    def complete(self, mass, z):
        return np.ones_like(mass)

    def incomplete(self, mass, z):
        return  np.full_like(mass, self.a, dtype=float)
    
comp= Completeness_model(0.7)
mock_test = MockGenerator(cosmo=cosmo, hmf=hmf, cluster_m=cluster_m,halo_mass_interval=[10**(13.0),1e16])
print(mock_test.sky_area())
halos_mock = mock_test.generate_halos_from_hmf(comp.incomplete)
print(len(halos_mock['is_detected']))
print(len(halos_mock[halos_mock['is_detected'] == 1]['z']))
print(len(halos_mock[halos_mock['is_detected'] == 0]['z']))
print(len(halos_mock[halos_mock['is_detected'] == 1]['z'])/len(halos_mock['is_detected']))

401.93756131445895
46393
32386
14007
0.6980794516414114


In [4]:
mock_test.cluster_mass_interval = [10**(13.0),1e15]
halos_mock["Mass_obs"] = halos_mock["Mass_obs"]
class Purity_model:
    
    """Class to generate completeness model for halo/cluster mocks."""
    def __init__(self, b):
        self.b = b

    
    def pure(self, mass, z):
        return np.ones_like(mass)
        
    def impure(self, mass, z):
        return  np.full_like(mass, self.b, dtype=float)

def scaling_relation(lnMobs, z):
    return lnMobs

pur= Purity_model(0.8)
clusters_mock = mock_test.generate_clusters_from_halos(halos_mock,2.0,pur.impure,scaling_relation)
#clusters_mock = mock_test.generate_clusters_from_halos(halos_mock,2.0,None,scaling_relation)
print(len(clusters_mock['parent_id']))
print(len(clusters_mock[clusters_mock['parent_id'] != 0]['z']))
print(len(clusters_mock[clusters_mock['parent_id'] == 0]['z']))
print(len(clusters_mock[clusters_mock['parent_id'] != 0]['z'])/len(clusters_mock['parent_id'] ))
clusters_mock['Mass'] = clusters_mock['Mass']/np.log(10) 

44644
32386
12258
0.7254278290475764


In [5]:
halo_coordinates = {"RA":"RA" , "DEC":"DEC" , "z":"z"}
halo_properties  = {"Mass":"halo_mass","halo_id":"halo_id", "R200c":"R200", 'is_detected': 'is_detected'}
detections_coordinates =  {"RA":"RA" , "DEC":"DEC" , "z":"z"}
detections_properties  = {"Mass":"cluster_mass","cluster_id":"cluster_id", "R200c":"R200", "parent_id": "parent_id"}
halo_member_properties  = {"halo_mass":"halo_mass","halo_id":"halo_id", "galaxy_id":"galaxy_id"}
cluster_member_properties  = {"cluster_mass":"cluster_mass","cluster_id":"cluster_id", "galaxy_id":"galaxy_id"}
halo_ids  = {"ID": "halo_id", "MemberID": "galaxy_id"}
detections_ids  = {"ID": "cluster_id", "MemberID": "galaxy_id"}

# Confusion Matrix

In [12]:
def get_ratios(metrics_result):
    """Safely extracts TP/FP/TN/FN even if the metrics function returns an error string."""
    if isinstance(metrics_result, dict) and 'ratios' in metrics_result:
        return metrics_result['ratios']
    return {'TP': np.nan, 'FP': np.nan, 'TN': np.nan, 'FN': np.nan}

dfs = {}
halos_mocks_mass = np.log(np.array([10**(13.0), 10**(13.3),10**(13.4) , 10**(13.7), 10**(14.0)]))

current_criteria_rows = []
# --- Main Loops ---
for mass in halos_mocks_mass:
    # 1. Data Prep
    halos_mock_cutted = halos_mock[halos_mock['Mass'] >= mass]
    halos_mock_cutted['Mass'] = halos_mock_cutted['Mass']/np.log(10)
    current_halo_size = len(halos_mock_cutted['Mass'])
    halo_galaxies = mock_test.generate_galaxies(catalog=halos_mock_cutted, object_type='halo')
    cluster_galaxies = mock_test.generate_galaxies(catalog=clusters_mock, object_type='cluster')
    halo_galaxies_shared = mock_test.get_common_galaxies_by_distance(halos_mock_cutted, cluster_galaxies, object_type='halo')
    cluster_galaxies_shared = mock_test.get_common_galaxies_by_distance(clusters_mock, halo_galaxies, object_type='cluster')
    combined_halos = vstack([halo_galaxies, halo_galaxies_shared])
    combined_clusters = vstack([cluster_galaxies, cluster_galaxies_shared])
    
    # 2. SkyMatch Initialization
    halos = sky_match.SkyMatch(
    query_data= halos_mock_cutted,
    query_coordinates=halo_coordinates,
    query_member_data=combined_halos,
    query_ids = halo_ids, 
    match_data=clusters_mock,
    match_coordinates=detections_coordinates,
    match_member_data=combined_clusters,
    match_ids = detections_ids
)
    #detections = halos.invert_query_match()
    
    halos_matched = halos.match_ID()
    unique_halos = halos_matched.select_best()

    #detections_matched = detections.match_ID()
    #unique_detections = detections_matched.select_best()
    
    # 5. Table Generation
    multiple_halos_table = halos_matched.to_table_complete(query_properties=halo_properties, match_properties=detections_properties)
    #multiple_detections_table = detections_matched.to_table_complete(query_properties=detections_properties, match_properties=halo_properties)
    
    unique_halos_table = halos_matched.to_table_best(best=unique_halos, query_properties=halo_properties, match_properties=detections_properties)
    #unique_detections_table = detections_matched.to_table_best(best=unique_detections, query_properties=detections_properties, match_properties=halo_properties)

    # 7. Metrics Calculation
    m_halo_r = get_ratios(calculate_split_metrics(multiple_halos_table, 'halo'))
    #m_clus_r = get_ratios(calculate_split_metrics(multiple_detections_table, 'cluster'))
    u_halo_r = get_ratios(calculate_catalog_metrics(unique_halos_table, halos_mock, 'halo'))
    #u_clus_r = get_ratios(calculate_catalog_metrics(unique_detections_table, clusters_mock, 'cluster'))

    # 8. Row Construction
    '''
    row = {
        'mass_limit': np.exp(mass),
        'halo_size': current_halo_size,
        'm_halo_TP': m_halo_r['TP'], 'm_halo_FP': m_halo_r['FP'], 'm_halo_TN': m_halo_r['TN'], 'm_halo_FN': m_halo_r['FN'],
        'm_clus_TP': m_clus_r['TP'], 'm_clus_FP': m_clus_r['FP'], 'm_clus_TN': m_clus_r['TN'], 'm_clus_FN': m_clus_r['FN'],
        'u_halo_TP': u_halo_r['TP'], 'u_halo_FP': u_halo_r['FP'], 'u_halo_TN': u_halo_r['TN'], 'u_halo_FN': u_halo_r['FN'],
        'u_clus_TP': u_clus_r['TP'], 'u_clus_FP': u_clus_r['FP'], 'u_clus_TN': u_clus_r['TN'], 'u_clus_FN': u_clus_r['FN'],
    }
    '''
    row = {
        'mass_limit': np.exp(mass),
        'halo_size': current_halo_size,
        'm_halo_TP': m_halo_r['TP'], 'm_halo_FP': m_halo_r['FP'], 'm_halo_TN': m_halo_r['TN'], 'm_halo_FN': m_halo_r['FN'],
        'u_halo_TP': u_halo_r['TP'], 'u_halo_FP': u_halo_r['FP'], 'u_halo_TN': u_halo_r['TN'], 'u_halo_FN': u_halo_r['FN'],
    }
    current_criteria_rows.append(row)
    print(f"  > Finished M = {np.exp(mass):.2e}")

# Store in dictionary
dfs =  pd.DataFrame(current_criteria_rows)

# --- Save to Multi-HDU FITS ---
hdul = fits.HDUList([fits.PrimaryHDU()]) # Empty primary


hdu = fits.BinTableHDU(Table.from_pandas(dfs))
hdu.name = name
hdul.append(hdu)

hdul.writeto('results_id.fits', overwrite=True)
print("\nAll masses complete. Results saved to 'results_id.fits'")

Generated 60720 galaxies from 46393 halo(s).
Generated 56003 galaxies from 44644 cluster(s).
  > Finished M = 1.00e+13
Generated 34109 galaxies from 21450 halo(s).
Generated 56010 galaxies from 44644 cluster(s).
  > Finished M = 2.00e+13
Generated 27745 galaxies from 16041 halo(s).
Generated 56103 galaxies from 44644 cluster(s).
  > Finished M = 2.51e+13
Generated 14928 galaxies from 6294 halo(s).
Generated 56038 galaxies from 44644 cluster(s).
  > Finished M = 5.01e+13
Generated 8013 galaxies from 2182 halo(s).
Generated 56078 galaxies from 44644 cluster(s).
  > Finished M = 1.00e+14

All masses complete. Results saved to 'results_id.fits'


In [13]:
dfs

,mass_limit,halo_size,m_halo_TP,m_halo_FP,m_halo_TN,m_halo_FN,u_halo_TP,u_halo_FP,u_halo_TN,u_halo_FN
0,1.000000e+13,46393,0.845559,0.154441,0.326906,0.673094,0.368487,0.631513,0.311832,0.688168
1,1.995262e+13,21450,0.901643,0.098357,0.343813,0.656187,0.752570,0.247430,0.317534,0.682466
2,2.511886e+13,16041,0.918777,0.081223,0.353277,0.646723,0.823088,0.176912,0.316524,0.683476
3,5.011872e+13,6294,0.958312,0.041688,0.411170,0.588830,0.928571,0.071429,0.312908,0.687092
4,1.000000e+14,2182,0.984314,0.015686,0.520654,0.479346,0.979000,0.021000,0.308329,0.691671
